# 📊 03. 데이터 통계 + 인사이트 리포트 — H&M 커머스

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

이 노트북은 **01_데이터이해_전처리 → 02_데이터_EDA → 03_데이터_통계(이 노트북)** 3파트 파이프라인의 **마지막**입니다. 02 에서 관찰만 하고 넘긴 질문("이 차이가 우연인지")을 여기서 통계로 확인하고, 프로젝트를 **인사이트 리포트**로 마무리합니다.

**구성**
1. **정제본 불러오기** — 01 이 만든 표를 그대로 이어 씁니다.
2. **가이드 — 통계 도구 지도** — day08(추론통계)·day09(가설검정·회귀)에서 배운 도구를 **전부** 모아 뒀습니다. **막힐 때 찾아 쓰는 참고서**입니다.
3. **선택 통계 미션 카탈로그** — ⚠️ **통계는 필수가 아닙니다.** 발제문의 필수 목표는 EDA·시각화·인사이트입니다. 이 절은 "차이가 우연인지까지 따지고 싶은 사람"을 위한 **선택**이며, **2~3개만 골라도 충분**합니다.
4. **최종 인사이트 리포트** — 이 프로젝트의 마무리, **필수**입니다.
5. **AI 협업 검증** — 오류가 심어진 AI 초안을 실제로 재계산해 잡아내는 실습입니다.
6. **제출 체크리스트** — 발제문 필수 목표를 다 담았는지 스스로 확인합니다.

## 1. 정제본 불러오기
01 이 저장한 정제본(`output/hm_clean.csv`)을 읽습니다. **원본 3개 테이블은 다시 읽지 않습니다.**

### 준비 ① 도구 챙기기 — 어떤 라이브러리를 왜 쓰나
이 노트북은 **코드를 주지 않습니다.** 각 절이 *무엇을* 해야 하는지와 *어떤 도구*를 쓰는지 알려 주니, 코드는 그것을 보고 **직접** 쓰세요. 먼저 아래 도구들을 불러오는 셀을 하나 만들어 두면 끝까지 씁니다.

| 무엇에 쓰나 | 어떤 도구(관례 별칭) | 쓰이는 절 |
| --- | --- | --- |
| 표·집계·수치 | `pandas`(`pd`) · `numpy`(`np`) | 전 절 |
| 그림 | `matplotlib.pyplot`(`plt`) · `seaborn`(`sns`) | 2-1 · 2-3 · 2-9 |
| 추정(표준오차·신뢰구간)·카이제곱 적합도 | `scipy` 의 `stats` | 2-1 · 2-7 ① |
| 가정 점검·검정·효과크기·사후검정 | `pingouin`(`pg`) | 2-3 ~ 2-7 |
| 비모수 사후검정(Dunn) | `scikit_posthocs`(`sp`) — pingouin 에 없습니다 | 2-6 |
| 회귀 | `statsmodels.formula.api`(`smf`) | 2-9 |
| 잔차 진단 | `statsmodels` 의 `durbin_watson` · `het_breuschpagan` | 2-9 |
| 카이제곱 사후분석(조정된 잔차) | `statsmodels` 의 `Table` | 2-7 ② |

**한글 폰트**도 여기서 정해 두세요 — 안 하면 그래프의 한글이 □□□ 로 깨집니다. `platform.system()` 으로 OS 를 보고 Windows 는 `'Malgun Gothic'`, macOS 는 `'AppleGothic'`, 리눅스·Colab 은 `'NanumGothic'` 을 `plt.rcParams['font.family']` 에 넣고, `plt.rcParams['axes.unicode_minus']` 를 `False` 로 두어 마이너스 부호 깨짐도 막습니다. `sns.set_theme(...)` 은 폰트 설정을 되돌리므로 **set_theme 에 `font=` 로 같은 폰트를 넘기거나, set_theme 뒤에 폰트를 다시 지정**하세요. 01·02 에서 쓴 준비 셀과 같은 내용이니 같은 프로젝트로 이어 쓰면 됩니다.

> 🔧 **막히면**: 경고 메시지가 너무 많으면 `warnings.filterwarnings('ignore')` 로 눌러 둘 수 있습니다(통계 라이브러리는 참고용 경고를 자주 냅니다).

### 준비 ② 정제본을 `df` 라는 이름으로 불러오기
- **무엇을**: 01 이 저장한 정제본 `output/hm_clean.csv` 를 `pd.read_csv` 로 읽습니다. **변수명은 `df`** 로 두세요 — 이 노트북의 모든 지침이 `df` 를 가정합니다. 읽은 뒤 `df.shape` 와 `df.head()` 로 무엇이 들어왔는지 먼저 확인합니다(데이터를 보지 않고 검정부터 하지 않습니다).
- **필요한 컬럼**: `t_dat`(거래일) · `age` · `price` · `sales_channel_id` · `club_member_status` · `product_group_name` · `customer_id`. 하나라도 없으면 **01 로 돌아가** 그 컬럼을 남긴 채 다시 저장하세요(원본 3개 테이블을 여기서 다시 읽지는 않습니다).
- **날짜형 복구**: CSV 를 거치면 `t_dat` 가 문자열로 풀립니다 → `pd.to_datetime` 으로 되돌리세요. 안 하면 뒤에서 `.dt` 접근이 실패합니다.
- **파생 컬럼(이름 그대로 만드세요)**: `month` = `t_dat` 의 월(`.dt.month`), `age_group` = 10년 단위 연령대(`age // 10 * 10`, 정수형). 요일까지 보고 싶으면 `weekday`(`.dt.dayofweek`)를 더해도 됩니다. 무거운 그림·계산에 쓸 표본이 필요하면 `df_s` 를 하나 만들어 두면 편합니다(3000행, `random_state=42` 로 고정하면 매번 같은 표본).

> ⚠️ **정제 규칙이 기준값을 바꿉니다** — 이 노트북에 적어 둔 기대값(예: 채널 t≈64.9, 5절 자가채점의 74.7%)은 **01 의 표준 규칙**을 적용한 **113,218행** 기준입니다: 중복 제거 · 나이 10~99세만 남김 · `price > 0` · `sales_channel_id` 는 `isin([1, 2])`(정의에 없는 코드 `0` 제외) · `club_member_status` 결측은 `fillna('Unknown')`. 여러분이 다른 규칙을 골랐다면 숫자가 조금씩 다를 수 있고 **그건 틀린 게 아닙니다**(01 에 이유를 적어 두었다면). 다만 **5절 자가채점만은** 위 규칙에 맞춰 두세요.

## 2. 가이드 — 통계 도구 지도 (막힐 때 여기)
day08(추론통계)·day09(가설검정·회귀)에서 배운 도구를 **목적별로 전부** 모아 뒀습니다. 각 절은 **무엇을 하는지 · 왜 그 방법인지(가정) · 어떤 함수로 · 결과를 어떻게 읽는지 · 흔한 함정**을 알려 줍니다 — **코드는 그것을 보고 직접 쓰세요.** 적어 둔 숫자는 위 정제본(113,218행)으로 실제 계산한 값이니 **여러분의 결과와 맞춰 보는 답지**로 쓰면 됩니다.

⚠️ **가이드에는 다 나오지만, 3절 미션은 이 중 2~3개만 실제로 해 보면 됩니다.**

### 2-1. 모집단·표본·표준오차·신뢰구간 (day08)
**이 절에서 할 일**: 표본으로 모집단을 추정하는 도구(CLT·표준오차·신뢰구간)를 복습합니다.

이 정제본(113,218행)도 사실 **전체 H&M 고객이 아니라 Kaggle 이 공개한 표본**입니다 — 우리가 보는 모든 평균·비율은 **모수(참값)의 추정치**입니다. day08 은 이 추정이 얼마나 믿을 만한지 재는 도구들입니다.

#### 중심극한정리(CLT) 를 눈으로 확인하기
- **무엇을 하나**: `price` 에서 크기 n 의 표본을 아주 많이(예: 2000번) 뽑아 **표본평균만 모아** 히스토그램(`sns.histplot`)을 그립니다. n = 1 · 30 · 100 세 경우를 `plt.subplots(1, 3, ...)` 로 나란히 놓고 모양을 비교하세요.
- **왜 이걸 보나**: 원자료(`price`)는 오른쪽으로 심하게 치우쳐 있는데도 **표본평균의 분포는** n 이 커지면 정규분포에 가까워집니다. 이것이 CLT 이고, 뒤에서 "정규성이 깨졌는데도 평균 비교 검정을 믿는" 근거가 됩니다.
- **어떤 도구**: 재현 가능한 난수는 `np.random.default_rng(42)`, 복원추출은 `rng.choice(모집단, size=(반복수, n), replace=True)` 로 한 번에 만들고 `.mean(axis=1)` 로 표본평균 배열을 얻습니다.
- **결과 읽는 법**: 각 그림 제목에 **경험적 SE**(표본평균들의 `.std()`)와 **이론 SE**(σ/√n — σ 는 모표준편차이므로 `.std(ddof=0)`)를 함께 적어, 두 값이 거의 같아지는지 보세요. n=100 이면 이미 종모양입니다.
- **함정**: n=1 그림은 원자료 분포 그 자체입니다(치우친 모양) — 이걸 "CLT 가 틀렸다"로 읽지 마세요. 그리고 CLT 가 말하는 것은 **표본평균**의 분포이지 원자료가 정규분포가 된다는 뜻이 아닙니다.

#### 표준오차(SE) — 표본이 커지면 추정이 왜 안정되나
- **무엇을 하나**: n=40 과 n=160(4배)에서 위와 같은 방식으로 표본평균의 표준편차(경험적 SE)를 구해 이론값 σ/√n 과 나란히 출력하세요.
- **어떤 도구**: 표본 하나에서 SE 를 바로 구할 때는 `stats.sem(표본)` 한 줄이면 됩니다.
- **결과 읽는 법**: n 을 4배로 늘리면 SE 는 약 √4 = 2배 작아집니다(실측 비율 ≈ 1.93). 그래서 표본이 클수록 평균 추정이 안정됩니다.
- **함정**: SE 는 "데이터가 얼마나 퍼져 있나"(표준편차)가 아니라 "**평균 추정치가** 얼마나 흔들리나"입니다. n 을 늘려도 표준편차는 그대로이고 SE 만 작아집니다.

#### 점추정 vs 구간추정 — 신뢰구간
- **무엇을 하나**: `price` 에서 200개를 뽑아 **점추정**(표본평균 — 숫자 하나)과 **구간추정**(95% 신뢰구간)을 구하고, 모평균(전체 `price` 평균)이 그 구간 안에 들어오는지 확인하세요.
- **어떤 도구**: `stats.t.interval(0.95, df=…, loc=…, scale=…)` — 자유도 `df` 는 표본 크기 − 1, `loc` 는 표본평균, `scale` 은 표준오차(`stats.sem(...)`)입니다. 표본이라 모표준편차를 모르므로 정규분포가 아니라 **t 분포**를 씁니다.
- **결과 읽는 법(중요)**: 올바른 해석은 "이 방법으로 100번 표본을 뽑아 구간을 만들면 **그중 약 95개가** 모평균을 포함한다"입니다.
- **흔한 오해**: "이 구간에 모평균이 있을 확률이 95%" ❌ — 모평균은 고정된 값이라 확률을 논할 대상이 아닙니다. 확률은 "구간을 만드는 절차" 쪽에 붙습니다.
- **주장 판단에 쓰기**: 누군가 "평균 단가는 0.05 다"라고 하면, 0.05 가 이 신뢰구간 밖인지 보면 됩니다(밖이면 이 데이터와 맞지 않는 주장). 신뢰구간과 검정은 동전의 양면입니다.
- **함정**: 신뢰구간은 표본이 커지면 좁아집니다 — 좁다고 "중요한 발견"이 아니라 "추정이 정밀하다"는 뜻일 뿐입니다.

**p-value 직관**: p-value 는 "**귀무가설(H₀)이 참이라고 가정했을 때**, 지금 관측한 값(또는 더 극단적인 값)이 나올 확률"입니다. 작을수록 "H₀ 가 맞다면 이런 결과가 나오기 어렵다"는 뜻이라 H₀ 를 의심하게 됩니다. **흔한 오해**: p-value 는 ❌ "H₀ 가 참일 확률"이 아니고, ❌ "결과가 우연이 아닐 확률"도 아니며, ❌ 작을수록 무조건 "효과가 크다"는 뜻도 아닙니다(뒤에서 계속 나올 "유의≠실질"의 출발점입니다).

> ✅ **여기까지 되면 통과**: 표본이 커질수록 표준오차가 왜 작아지는지, 신뢰구간이 무엇을 보장하는지 말로 설명할 수 있으면 됩니다.

### 2-2. 가설검정 프레임 (day09)
**이 절에서 할 일**: 가설검정의 공통 언어(H₀/H₁·α·오류·양측/단측)를 정리합니다. 이후 모든 검정이 이 틀을 그대로 씁니다.

| 용어 | 뜻 |
| --- | --- |
| **귀무가설 H₀** | "차이(효과) 없다" — 기본 입장 |
| **대립가설 H₁** | "차이(효과) 있다" — 보이고 싶은 주장 |
| **유의수준 α** | 우연을 진짜라고 오해할 위험의 상한(보통 0.05) |
| **검정통계량** | t·F·χ² 등 — 데이터를 H₀ 아래에서 얼마나 벗어났는지로 요약한 값 |
| **p-value** | H₀ 가 참일 때 이 값(또는 더 극단)이 나올 확률. **p < α** 면 H₀ 기각 |
| **1종 오류** | H₀ 가 참인데 기각(잘못된 "차이 있다") — 확률 α |
| **2종 오류** | H₀ 가 거짓인데 기각 못함(못 알아챈 진짜 차이) — 확률 β |

**양측 vs 단측**: H₁ 이 "다르다(≠)"면 **양측**(기본값, `alternative='two-sided'`), "더 크다/작다"면 **단측**(`'greater'`/`'less'`)입니다. ⚠️ **방향을 반대로 잡으면 결론이 뒤집힙니다** — 같은 데이터에서 옳은 방향의 단측 p 는 양측 p 의 정확히 절반이지만, 틀린 방향으로 잡으면 p 가 1 에 가까워져 유의하지 않게 됩니다. **방향은 데이터를 보기 전에 정해야** 합니다(보고 나서 유리한 쪽으로 바꾸면 1종 오류가 부풀어 오릅니다). 이 노트북은 특별한 말이 없으면 **양측**을 씁니다.

> ✅ **여기까지 되면 통과**: H₀/H₁ 을 스스로 세울 수 있고, 단측을 잘못 골랐을 때 결론이 어떻게 뒤집히는지 설명할 수 있으면 됩니다.

### 2-3. 가정 점검 절차 + 검정 선택표
**이 절에서 할 일**: 평균을 비교하기 전에 **① 정규성 → ② 등분산**을 확인하고, 그 결과로 어떤 검정을 쓸지 고릅니다. 이후 모든 평균 비교 미션이 이 절차를 그대로 따릅니다.

#### ① 집단별 정규성 — 각 집단 안에서 값이 정규분포에 가까운가
- **무엇을 하나**: 연령대(`age_group`)별로 단가(`price`)의 정규성을 검정하고, 대표로 한 집단(예: 30대)만 **Q-Q Plot** 도 그려 보세요.
- **왜 먼저 보나**: 평균을 비교하는 검정(t·ANOVA)은 "각 집단이 대략 정규분포"를 가정합니다. 가정을 보지 않고 검정하면 p-value 자체를 믿을 수 없습니다.
- **어떤 도구**: `pg.normality(data=…, dv=…, group=…)` — 집단마다 한 줄씩 W 통계량·p·`normal`(True/False)이 나옵니다. 그림은 `pg.qqplot(한 집단의 값, dist='norm', ax=…)` 로, 새 Axes 를 `plt.subplots()` 로 하나 만들어 넘기세요.
- **결과 읽는 법**: 표에서는 `normal` 열만 봐도 됩니다(p<0.05 면 정규성 기각). Q-Q Plot 은 점이 대각선을 따라 놓이면 정규에 가깝고, **오른쪽 끝이 위로 휘면 오른쪽 꼬리가 긴 분포**입니다. 이 데이터는 **모든 연령대에서 정규성이 기각**됩니다(단가가 치우쳐서). 다만 집단마다 표본이 수천~수만 건이라 CLT 로 평균 비교는 견딥니다 → 그렇다면 갈림길은 다음의 등분산입니다.
- **함정**: 대표본에서는 아주 작은 비정규성도 검정이 잡아냅니다 → "정규성 기각 = 무조건 비모수"가 아닙니다. **표본 크기와 함께** 판단하세요(그래서 그림을 같이 봅니다).

#### ② 등분산 — 집단들의 퍼짐이 비슷한가
- **어떤 도구**: `pg.homoscedasticity(data=…, dv=…, group=…)`(기본값이 Levene 검정) → 결과의 `equal_var` 열이 `False` 면 **등분산 가정이 깨진 것**입니다.
- **결과 읽는 법**: 연령대별 단가는 Levene p ≈ 1.2e-34 로 **등분산 위배**입니다. 이 한 줄이 아래 선택표에서 **어느 열을 볼지**를 결정합니다(같은 데이터라도 검정 종류가 바뀝니다).
- **함정**: Levene 은 "분산이 다르다"까지만 말합니다. 어느 집단의 분산이 큰지는 `groupby('age_group')['price'].std()` 처럼 따로 봐야 합니다.

**③ 검정 선택표** — 위 ①②의 결과에 따라 아래 표로 검정을 고릅니다(대표본이면 정규성 위반은 CLT 로 어느 정도 버티지만, 등분산 위반은 검정 종류를 바꿉니다).

| 데이터 유형 | 등분산 만족 | 등분산 위반 | 정규성 심하게 위반 + 소표본 |
| --- | --- | --- | --- |
| 2집단 독립 | Student t-검정 | **Welch t-검정** | Mann-Whitney U (`pg.mwu`) |
| 2집단 대응(같은 대상) | 대응 t-검정 | (해당 없음) | Wilcoxon 부호순위 (`pg.wilcoxon`) |
| 3집단 이상 | ANOVA + Tukey | **Welch-ANOVA + Games-Howell** | Kruskal-Wallis + Dunn(`sp.posthoc_dunn`) |
| 범주 × 범주(연관) | 카이제곱 독립성(+Cramér's V, 조정잔차) | 〃 | 〃 |
| 범주 1개(비율 확인) | 카이제곱 적합도 | 〃 | 〃 |

> ✅ **여기까지 되면 통과**: 정규성·등분산 결과를 보고 이 표에서 검정을 정확히 짚어낼 수 있으면 됩니다.

### 2-4. 평균 비교 — t-검정 3종 + 효과크기
**이 절에서 할 일**: 상황별 t-검정 세 가지(1표본·대응·독립)를 실행하고, p-value 와 함께 **Cohen's d(효과크기)** 를 항상 같이 봅니다.

세 가지 모두 함수는 하나입니다 — `pg.ttest(...)` 에 **무엇을 넣는지**로 종류가 갈립니다.

**결과 표에서 볼 것 세 개**: `p_val`(우연인가) · `cohen_d`(얼마나 큰가 — 부호 없는 |d|) · 실제 평균값(어느 방향인가). **참고만**: `T`·`dof`·`CI95`·`power`. **지금은 무시**: `BF10`(베이즈인자).

#### ① 1표본 t-검정 — "평균 단가가 0.025 다"라는 주장 검증
- **무엇을 검정하나**: H₀ "모평균 = 0.025" vs H₁ "다르다"(양측).
- **어떤 도구**: `pg.ttest(값들, 비교기준값)` — 첫 인자에 `price` 열, 둘째 인자에 `0.025`.
- **결과 읽는 법**: 실제 평균 ≈ 0.0276, T ≈ 47.6, d ≈ 0.141 → **유의하지만 효과는 작습니다**. "주장과 다르다"는 말할 수 있지만 "많이 다르다"는 말할 수 없습니다.

#### ② 대응(paired) t-검정 — 같은 상품군 안에서 온라인 vs 오프라인
- **왜 대응인가**: 상품군마다 가격대가 다릅니다. 같은 상품군끼리 **짝을 지어 차이만** 보면 상품군 효과가 상쇄되고 채널 효과만 남습니다.
- **짝 만들기**: `pivot_table(index=…, columns=…, values=…, aggfunc='mean')` 로 상품군(행) × 채널(열) 평균가 표를 만들고 `dropna()` 로 한쪽 채널만 있는 상품군을 떨어냅니다 — `index` 는 `product_group_name`, `columns` 는 `sales_channel_id`, `values` 는 `price`. 양쪽에 다 있는 상품군 12개가 짝이 됩니다. 이 표는 `piv` 로 두면 2-5 에서 다시 씁니다.
- **어떤 도구**: `pg.ttest(온라인_열, 오프라인_열, paired=True)` — 열 이름은 채널 코드 그대로 `2`(온라인)·`1`(오프라인)입니다.
- **결과 읽는 법**: T ≈ 4.27, p ≈ 0.0013, d ≈ 0.499(중간) → 같은 상품군이어도 온라인이 평균적으로 더 비쌉니다.
- **함정**: 길이가 같다고 대응이 아닙니다 — **같은 대상**(여기선 같은 상품군)으로 짝지어졌을 때만 `paired=True` 를 씁니다. 12쌍은 표본이 작으니 비모수(2-5)도 함께 봅니다.

#### ③ 독립 2표본 t-검정 — 온라인 거래 전체 vs 오프라인 거래 전체
- **무엇을 준비하나**: 채널로 나눈 두 시리즈를 만드세요 — `online`(`sales_channel_id == 2` 의 `price`)·`offline`(`== 1`). 이 둘도 2-5 에서 재사용합니다.
- **어떤 도구**: `pg.ttest(a, b, correction=True)` — `correction=True` 가 **Welch**(등분산 미가정)입니다. 2-3 에서 등분산이 깨진 것을 확인했으니 이쪽을 씁니다(등분산이면 `False`).
- **결과 읽는 법**: 온라인 0.0299 vs 오프라인 0.0228(약 1.3배), T ≈ 64.9, d ≈ 0.376(작은~중간).
- **함정**: 표본이 11만 건이면 **작은 차이도 p ≈ 0** 이 됩니다. p 만 보고 "큰 차이"라고 쓰지 말고 항상 d 를 함께 보고하세요.

> ✅ **여기까지 되면 통과**: 세 t-검정이 서로 다른 상황(주장 검증·짝지은 비교·두 독립 집단)에 쓰인다는 것을 구분하고, `correction=True` 가 왜 필요했는지(2-3 의 등분산 결과) 설명할 수 있으면 됩니다.

### 2-5. 모수 vs 비모수 — 정규성이 심하게 깨졌을 때
**이 절에서 할 일**: 표본이 작거나 정규성이 심하게 깨졌을 때 쓰는 **순위 기반 검정**을 t-검정과 짝지어 봅니다. (이 데이터는 표본이 커 t-검정도 견디지만, 방법 자체를 알아 둡니다.)

#### Mann-Whitney U — 독립 2표본 t-검정의 비모수 짝
- **무엇을 검정하나**: 평균이 아니라 **순위**를 비교합니다 → 정규성 가정이 필요 없고 극단값에 둔감합니다.
- **어떤 도구**: `pg.mwu(a, b)` — 전체 데이터는 무거우니 각 채널에서 2000개씩 뽑아 쓰세요(`sample(2000, random_state=42)` 로 고정하면 재현됩니다). ⚠️ **인자 순서가 U 값과 `RBC` 의 부호를 정합니다** — 온라인을 첫 인자로 두면 "온라인이 더 큼"이 양수로 나옵니다.
- **결과 읽는 법**: `RBC`(방향이 있는 효과크기, -1~1) ≈ 0.249, `CLES`(한쪽에서 뽑은 값이 다른 쪽보다 클 확률) ≈ 0.624 → "온라인에서 뽑은 거래가 더 비쌀 확률이 약 62%"로 읽습니다.

#### Wilcoxon 부호순위 — 대응 t-검정의 비모수 짝
- **어떤 도구**: `pg.wilcoxon(x, y)` — 2-4 ② 에서 만든 `piv` 의 12쌍을 그대로 넣으세요.
- **결과 읽는 법**: p ≈ 0.00098 → 대응 t-검정(p ≈ 0.0013)과 **결론이 같습니다**. 방법을 바꿔도 같은 결론이면 그 발견은 더 믿을 만합니다.
- **함정**: 비모수는 "가정이 없다"가 아니라 "**정규성** 가정이 없다"입니다(독립성·분포 모양 가정은 남습니다). 또 표본이 충분히 크면 모수 검정이 더 힘이 좋으니, 대표본에서 굳이 비모수만 쓸 이유는 없습니다.

> ✅ **여기까지 되면 통과**: `RBC`·`CLES` 가 무엇을 뜻하는지 말할 수 있고, 같은 데이터에서 모수 검정과 비모수 검정의 결론이 (대개) 일치한다는 것을 확인했으면 됩니다.

### 2-6. 3집단 이상 비교 — ANOVA/Welch-ANOVA + 사후검정, 비모수 Kruskal+Dunn
**이 절에서 할 일**: 02 에서 넘어온 질문 — **"연령대별 평균 단가 차이가 통계적으로 유의한가?"** — 를 정식 검정으로 확인합니다. 2-3 에서 이미 **정규성 위반 + 등분산 위반**(Levene p=1.2e-34)을 확인했으므로 **Welch-ANOVA** 가 적절합니다.

#### ① 3집단 이상 평균 비교 — ANOVA vs Welch-ANOVA
- **무엇을 검정하나**: H₀ "모든 연령대의 평균 단가가 같다" vs H₁ "적어도 한 집단이 다르다". `dv` 는 `price`, `between` 은 `age_group`(9집단)입니다.
- **왜 t-검정을 여러 번 쓰지 않나**: 9집단을 두 개씩 36번 t-검정하면 우연히 유의해질 확률(1종 오류)이 부풀어 오릅니다. ANOVA 는 한 번에 봅니다.
- **어떤 도구**: `pg.anova(data=…, dv=…, between=…, detailed=True)` 와 `pg.welch_anova(data=…, dv=…, between=…)` 를 나란히 돌려 보세요.
- **결과 읽는 법**: 볼 것은 `p_unc`(유의한가)와 `np2`(= η², 얼마나 큰가)입니다. `F` 와 자유도는 참고, `SS`·`MS` 는 중간 계산이니 **지금은 무시**하세요. 일반 F ≈ 52.8 / Welch F ≈ 55.4(p ≈ 5e-37), η² ≈ 0.004 → **등분산이 깨졌으므로 Welch 쪽을 채택**해 보고합니다.
- **함정**: ANOVA 는 "어딘가 다르다"까지만 말합니다. 어느 연령대끼리 다른지는 사후검정이 답합니다.

#### ② 사후검정 — 어느 쌍이 다른가
- **어떤 도구**: 등분산 가정이면 `pg.pairwise_tukey(data=…, dv=…, between=…)`(p 열 이름 `p_tukey`), 등분산 미가정이면 `pg.pairwise_gameshowell(data=…, dv=…, between=…)`(p 열 이름 `pval`). 이번 데이터는 **Games-Howell** 이 맞습니다.
- **결과 읽는 법**: 36쌍 중 `p < 0.05` 인 쌍의 개수를 세어 보세요 — Tukey 17쌍 · Games-Howell 18쌍. 표에서 `diff`(차이)와 `hedges`(효과크기)도 함께 보면 "유의하지만 미미한 쌍"이 보입니다.
- **함정**: 사후검정은 다중비교 보정을 이미 포함합니다 — 여기서 나온 p 를 다시 보정하지 마세요.

#### ③ 비모수 대안 — Kruskal-Wallis + Dunn
- **어떤 도구**: `pg.kruskal(data=…, dv=…, between=…)` → H ≈ 425, p ≈ 8.5e-87. 사후검정은 pingouin 에 없어 `sp.posthoc_dunn(…, val_col=…, group_col=…, p_adjust='bonferroni')` 를 씁니다.
- **결과 읽는 법**: Dunn 은 **대칭 행렬**로 반환되므로 쌍을 셀 때 위삼각만 세야 합니다(`np.triu_indices_from(…, k=1)`). 유의한 쌍 17/36 → **Games-Howell 과 거의 같은 결론**입니다(경로가 달라도 수렴).
- **함정**: 모수·비모수 결과가 엇갈리면 어느 한쪽을 골라 보고하지 말고 **왜 갈렸는지**(표본 크기·극단값·분포 모양)를 함께 쓰세요.

**결론**: F=55.4(Welch), p≈0 로 연령대별 단가 차이는 **유의**하지만 η²=0.0037(0.4%)로 **실질적 효과는 매우 작습니다** — 02 의 질문에 대한 답은 "유의하지만 미미하다"입니다. 사후검정(Games-Howell)·비모수(Dunn) 모두 36쌍 중 약 절반이 유의해 **비슷한 결론에 수렴**합니다.

> ✅ **여기까지 되면 통과**: 등분산이 깨졌을 때 왜 Welch-ANOVA·Games-Howell 을 쓰는지 설명하고, η² 가 작다는 것과 p 가 유의하다는 것이 왜 모순이 아닌지 말할 수 있으면 됩니다.

### 2-7. 범주형 관계 — 카이제곱
**이 절에서 할 일**: **적합도**(한 범주형 변수가 특정 비율을 따르는지)와 **독립성**(두 범주형 변수가 관련 있는지)을 구분해 씁니다. 02 에서 넘어온 두 번째 질문 — **"연령대와 채널 선호가 독립적인가?"** — 도 여기서 답합니다.

#### ① 적합도 검정 — 한 범주형 변수가 특정 비율을 따르나
- **무엇을 검정하나**: H₀ "온·오프라인 거래 비중이 50:50 이다". 변수 **하나**의 분포에 대한 검정입니다.
- **어떤 도구**: 관측도수는 `df['sales_channel_id'].value_counts().sort_index()`, 기대도수는 전체 합을 반으로 나눠 만들고 `stats.chisquare(관측도수, 기대도수)` 에 넣습니다(적합도는 pingouin 에 없어 scipy 를 씁니다).
- **결과 읽는 법**: χ² ≈ 16768.7, p ≈ 0 → 50:50 가정은 기각됩니다(온라인이 훨씬 많습니다).
- **함정**: 기대도수는 **비율이 아니라 도수**여야 하고 **관측 합계와 같아야** 합니다(안 맞으면 scipy 가 에러를 냅니다). 그리고 이 검정은 "두 변수의 관계"가 아니라 "한 변수의 분포" 이야기입니다.

#### ② 독립성 검정 — 두 범주형 변수가 서로 관련 있나 (02 의 두 번째 질문)
- **무엇을 검정하나**: H₀ "연령대와 채널 선택은 독립이다(서로 무관하다)".
- **어떤 도구**: `pg.chi2_independence(data=…, x=…, y=…)` — **세 개를 반환**합니다(기대빈도 · 관측빈도 · 검정결과표). 검정결과표에는 여러 변형이 나오는데 **`pearson` 줄만** 보면 됩니다.
- **가정 점검이 먼저**: 카이제곱은 **모든 칸의 기대빈도가 5 이상**이어야 믿을 수 있습니다. 반환된 기대빈도표의 최솟값(`.values.min()`)을 확인하세요 — 원본 9개 연령대로는 **1.85** 로 가정 위배입니다(90대 6건·80대 60건처럼 표본이 아주 작은 연령대 때문). 이럴 때는 **작은 범주를 묶습니다** — 예: 70대 이상을 하나로(`clip(upper=70)`) 만든 새 컬럼 `age_group2` 로 다시 검정하면 최소 기대빈도가 5 를 넘습니다.
- **결과 읽는 법**: χ² ≈ 704.5, p ≈ 0 → 독립이 아닙니다. 다만 `cramer`(Cramér's V) ≈ 0.079 로 **연관 강도는 약합니다**.
- **어느 칸이 튀는지(조정된 잔차)**: `Table(관측빈도.values).standardized_resids` 를 관측표의 index·columns 로 감싼 DataFrame 으로 보면 칸별로 기대보다 많은지/적은지가 표준화된 값으로 나옵니다. **±2 를 넘으면 눈에 띄는 칸**이고, 여기서는 30대의 온라인 칸이 +20.5 로 가장 두드러집니다.
- **함정**: 카이제곱은 "관계가 있다"까지만 말합니다 — **방향은 말해 주지 않습니다**. 방향은 잔차나 비율표(`pd.crosstab(..., normalize='index')`)로 따로 봐야 합니다. 그리고 범주를 묶는 것은 "숫자 맞추기"가 아니라 **가정을 지키기 위한 조치**이니, 묶은 이유를 리포트에 적으세요.

**결론**: 연령대와 채널은 **독립이 아니다**(χ²=704.5, p≈0)지만 Cramér's V=0.079 로 **연관은 약합니다** — 02 의 두 번째 질문에 대한 답도 "유의하지만 약하다"입니다. 그리고 **가정 점검(기대빈도 ≥5)이 실전에서 왜 중요한지**도 함께 보였습니다 — 원본 연령대 그대로는 최소 기대빈도가 1.85 로 가정이 깨져, 90대·80대를 '70+' 로 묶어야 검정을 믿을 수 있었습니다.

> ✅ **여기까지 되면 통과**: 적합도와 독립성의 차이를 설명하고, 기대빈도가 5 미만일 때 왜 결과를 그대로 믿으면 안 되는지 말할 수 있으면 됩니다.

### 2-8. 효과크기 기준표 — "유의"와 "실질"은 다르다
**이 절에서 할 일**: p-value 는 "우연이 아니다"만 말합니다. **차이가 얼마나 큰지**는 아래 효과크기로 따로 봅니다. 표본이 클수록(이 데이터처럼 11만 건) 아주 작은 차이도 p 가 유의해지므로, **효과크기 없이 p 만 보는 것은 위험**합니다.

| 효과크기 | 작음 | 중간 | 큼 | 쓰이는 곳 |
| --- | --- | --- | --- | --- |
| Cohen's d | 0.2 | 0.5 | 0.8 | t-검정 |
| η²(에타제곱) | 0.01 | 0.06 | 0.14 | ANOVA |
| Cramér's V | 0.1 | 0.3 | 0.5 | 카이제곱(2x2 기준, 자유도 클수록 기준 낮아짐) |
| RBC(비모수) | 0.1 | 0.3 | 0.5 | Mann-Whitney U |

![효과 크기 해석 가이드](images/효과크기_해석.png)

> ✅ **여기까지 되면 통과**: 2-4~2-7 에서 나온 효과크기 숫자들을 이 표에 놓고 "작다/중간/크다"를 스스로 분류할 수 있으면 됩니다.

### 2-9. 회귀 — 상관에서 다중회귀·LINE 진단까지
**이 절에서 할 일**: "무엇이 고객의 총구매액을 예측하는가"를 상관 → 단순회귀 → 다중회귀 → 범주형 더미까지 확장하고, 그 회귀를 믿어도 되는지 **LINE 4가정**으로 진단합니다.

#### 상관 → 단순회귀 → 다중회귀
- **먼저 분석 단위를 바꿉니다**: 지금까지는 "거래 1건"이 한 행이었지만, "고객의 총구매액"을 설명하려면 **고객 1명이 한 행**이어야 합니다. `df.groupby('customer_id').agg(...)` 로 고객 단위 표를 만들고 이름은 `cust` 로 두세요(끝에 `reset_index()`). 필요한 열 네 개는 **이름 그대로**: `total`(`price` 합) · `n`(거래 건수) · `age`(첫 값) · `online_ratio`(채널 2 의 비율 — `lambda` 로 "채널이 2 인 비율"을 계산하면 됩니다).
- **상관 먼저**: `cust['age'].corr(cust['total'])` ≈ 0.054 — 거의 관계가 없습니다. 회귀 전에 상관을 보는 습관을 들이세요.
- **단순회귀**(`m_simple`): `smf.ols('total ~ age', data=…).fit()` → `.rsquared` ≈ 0.0026.
- **다중회귀**(`m_multi`): `smf.ols('total ~ age + n + online_ratio', data=…).fit()` → `.rsquared` ≈ 0.382, `.rsquared_adj` ≈ 0.382. 계수·p 는 `.summary()` 로 한 번에 봅니다.
- **결과 읽는 법**: 설명변수를 넣을수록 R² 는 **늘기만** 합니다 → 모형 비교는 변수 개수에 벌점을 주는 **Adj R²** 로 하세요. 여기서는 구매빈도 `n` 이 설명력의 대부분을 끌어올립니다. `.summary()` 에서 볼 것은 계수(coef)·p 값·R²/Adj R² 이고, `std err`·`t`·`AIC`·`BIC`·`Omnibus`·`Cond. No.` 는 **지금은 무시**해도 됩니다.
- **함정 ①**: 설명변수끼리 강하게 상관되면(**다중공선성**) 계수가 불안정해지고 부호가 뒤집히기도 합니다 — 넣을 변수끼리 `.corr()` 로 먼저 확인하고, 더 정확히 보려면 9일차에서 쓴 **VIF**(`statsmodels.stats.outliers_influence.variance_inflation_factor`)로 변수마다 값을 구하세요. **VIF 10 이상**이면 그 변수는 다른 설명변수들로 거의 설명된다는 뜻이라 둘 중 하나를 빼는 것을 검토합니다(자세한 절차는 9일차 `참고_다중공선성.ipynb`).
- **함정 ②**: `total` 은 `price` 의 합인데 `n`(구매 건수)을 설명변수로 넣으면 "많이 사면 많이 쓴다"에 가까운 **동어반복**입니다. R² 가 높다고 새로운 발견이 아닙니다 — 무엇을 설명하려는 모형인지 먼저 정하세요.

#### 범주형 변수를 회귀에 넣기 — 더미(원-핫)
- **무엇을 하나**: 멤버십 상태(`club_member_status`)처럼 **글자로 된 변수**를 회귀에 넣습니다. 고객 단위 표를 하나 더 만들어(예: `cust_club` — `total` 과 `club` 두 열) 쓰세요.
- **어떤 도구**: `smf.ols('total ~ C(club)', data=…).fit()` — 식 안에서 `C()` 로 감싸면 더미(k-1개)를 **자동 생성**합니다(직접 원-핫 표를 만들지 않아도 됩니다).
- **결과 읽는 법**: `.params` 를 보면 첫 범주(여기선 `ACTIVE`)가 **기준 범주**로 빠져 있습니다. 나머지 계수는 절대 수준이 아니라 **"기준 범주와의 차이"** 입니다 — "PRE-CREATE 계수 = ACTIVE 대비 총구매액 차이"로 읽으세요.
- **함정**: 모든 범주를 다 더미로 넣으면 열끼리 완전히 겹쳐(**더미 변수 함정 = 완전 다중공선성**) 모형이 풀리지 않습니다 — 그래서 k-1 개만 씁니다. 기준 범주가 무엇인지 밝히지 않은 채 계수를 보고하면 해석이 달라지니 리포트에 꼭 적으세요.

#### LINE 4가정 잔차 진단 — 이 회귀를 믿어도 되는가
회귀는 돌리면 항상 숫자가 나옵니다. **믿을 수 있는지는 잔차**(`m_multi.resid`)가 알려 줍니다. 네 가정을 하나씩 확인하세요.

| 글자 | 가정 | 어떻게 보나 | 이 데이터 결과 |
| --- | --- | --- | --- |
| **L**inearity | 관계가 직선인가 | 잔차 vs 적합값 산점도(`m_multi.fittedvalues` × `m_multi.resid`)에 0 기준선(`axhline`)을 긋고, 곡선 무늬가 없는지 | 뚜렷한 곡선은 없음 |
| **I**ndependence | 잔차가 서로 독립인가 | `durbin_watson(m_multi.resid)` — 2 근처면 OK | ≈ 2.00 (OK) |
| **N**ormality | 잔차가 정규분포인가 | `pg.qqplot(m_multi.resid, dist='norm', ax=…)` | 꼬리가 휘어 위배 |
| **E**qual variance | 잔차 퍼짐이 일정한가 | 같은 산점도가 나팔 모양인지 + `het_breuschpagan(m_multi.resid, m_multi.model.exog)` 의 p | p ≈ 0 → 위배 |

- **결과 읽는 법**: 그림 두 개(잔차 산점도 · 잔차 Q-Q)와 숫자 두 개(Durbin-Watson · Breusch-Pagan p)면 충분합니다. 위배가 있으면 "이 모형은 쓸 수 없다"가 아니라 **"계수의 방향은 참고할 수 있지만 p-value·신뢰구간의 소수점까지는 과신하지 않는다"** 로 결론을 조절하세요(실무에서는 `log(total)` 로 다시 적합하거나 강건표준오차를 씁니다).
- **함정**: 대표본에서는 진단 검정(Breusch-Pagan 등)도 거의 항상 유의해집니다 → **그림과 함께** 판단하세요. 그림을 그릴 때는 셀마다 `plt.subplots()` 로 새 Axes 를 만들어야 앞 그림에 겹쳐 그려지지 않습니다.

**외삽(extrapolation) 위험**: 이 회귀선은 **관측된 나이·구매빈도 범위 안에서만** 믿을 수 있습니다. 데이터에 없는 극단값(예: 200세, 구매 1000회)을 넣어 예측하면 회귀식은 무너집니다. **유의≠실질**: 위 계수들의 p 는 유의하지만 단순회귀의 R²=0.0026 은 "나이가 총구매액의 0.26%만 설명한다"는 뜻입니다 — 표본이 커서(93,053명) 아주 작은 관계도 유의해진 것이지, 관계가 크다는 뜻이 아닙니다.

> ✅ **여기까지 되면 통과**: Adj R² 가 왜 필요한지, 더미의 '기준 범주'가 무엇인지, LINE 네 글자가 각각 무엇을 뜻하는지 설명할 수 있으면 됩니다.

## 3. 선택 통계 미션 카탈로그
⚠️ **다시 한 번 — 통계는 필수가 아닙니다.** 발제문의 필수 목표는 EDA·시각화·인사이트입니다. 아래는 **2절 가이드를 직접 손으로 해 보는 연습**이며, **2~3개만 골라도 충분**합니다.

| # | 미션 | 난이도 | 도구 | 예상 시간 |
| --- | --- | --- | --- | --- |
| A | 채널별 단가 차이 검정 | ★★ | `pg.homoscedasticity`·`pg.ttest` | 15분 |
| B | 연령대별 단가 차이 + 사후검정 *(02 연계)* | ★★★ | `pg.anova`/`welch_anova`·사후검정 | 20분 |
| C | 연령대 × 채널 독립성 검정 *(02 연계)* | ★★★ | `pg.chi2_independence`·조정잔차 | 20분 |
| D | 멤버십별 단가 차이 검정 | ★★★ | `pg.anova`·`pg.pairwise_tukey` | 20분 |
| E | 고객 총구매액 회귀 + LINE 진단 | ★★★ | `smf.ols`·잔차 진단 | 25분 |

### 미션 A — 채널별 단가 차이 검정
온라인·오프라인의 평균 단가가 정말 다른지 **가정 점검 → 검정 선택 → 실행**의 전 과정을 **직접** 작성해 보세요(가이드 2-4 ③과 같은 질문이지만, 이번엔 스스로).

> ✅ **이런 결과면 잘 된 것**: 등분산 점검 결과에 맞는 t-검정을 고르면 T 값이 약 64.9 근처로 나옵니다.
> 🔧 **막히면 볼 곳**: 가이드 2-3(가정 점검)·2-4(t-검정)

In [ ]:
# 여기에 코드를 작성하세요

### 미션 B — 연령대별 단가 차이 + 사후검정
02 의 질문 — "연령대별 평균 단가 차이가 통계적으로 유의한가?" — 를 **직접** 검정하세요. 정규성·등분산을 점검하고, 그 결과에 맞는 ANOVA(또는 Welch-ANOVA)와 사후검정을 실행하세요.

> ✅ **이런 결과면 잘 된 것**: F 값이 약 55(Welch) 근처, η² 는 0.01 미만(매우 작음)으로 나오고, 사후검정에서 36쌍 중 절반가량이 유의하게 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-3(가정 점검 선택표)·2-6(ANOVA/Welch-ANOVA)

In [ ]:
# 여기에 코드를 작성하세요

### 미션 C — 연령대 × 채널 독립성 검정
02 의 두 번째 질문 — "연령대와 채널 선호가 독립적인가?" — 를 카이제곱 독립성 검정으로 확인하세요. **기대빈도가 5 이상인지** 먼저 확인하고, 부족하면 연령대를 묶어 다시 검정하세요. Cramér's V 와 조정된 잔차까지 확인하세요.

> ✅ **이런 결과면 잘 된 것**: 그룹핑 후 최소 기대빈도가 5 이상이 되고, χ² 는 유의(p≈0)하지만 Cramér's V 는 0.1 미만(약함)으로 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-7(카이제곱)

In [ ]:
# 여기에 코드를 작성하세요

### 미션 D — 멤버십별 단가 차이 검정
채널·연령대와는 다른 축인 **멤버십 상태(`club_member_status`)** 별로 단가가 다른지 검정하세요. 이번엔 등분산이 만족되는 경우인지 직접 확인해 보세요(가이드 2-6 은 연령대만 다뤘습니다).

> ✅ **이런 결과면 잘 된 것**: 등분산이 만족되어 일반 ANOVA를 쓰면 F 는 약 9 근처, p 는 유의하지만 η² 는 0.001 미만(극히 작음)으로 나오고, 사후검정에서는 딱 한 쌍만 유의하게 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-3(가정 점검)·2-6(ANOVA)

In [ ]:
# 여기에 코드를 작성하세요

### 미션 E — 고객 총구매액 회귀 + LINE 진단
고객별 총구매액을 나이·구매빈도·온라인비중으로 설명하는 **다중회귀**를 적합하고, LINE 4가정을 잔차로 진단하세요(가이드 2-9 의 지침대로 만든 `cust`·`m_multi` 를 그대로 이어 써도 됩니다).

> ✅ **이런 결과면 잘 된 것**: R² 는 약 0.38, 세 계수 모두 p<0.001 로 유의하고, Durbin-Watson 은 2 근처(독립성 OK)지만 Breusch-Pagan p 는 사실상 0(등분산 위배)으로 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-9(회귀·LINE 진단)

In [ ]:
# 여기에 코드를 작성하세요

## 4. 최종 인사이트 리포트 (필수)
여기부터는 **선택이 아닙니다.** 01→02→03 에서 본 것을 하나의 리포트로 묶는, 이 프로젝트의 마무리입니다.

**리포트 뼈대 — 다섯 조각**
1. **무엇을 물었나** — 이 분석의 질문·가설(00 에서 세운 비즈니스 목표와 연결).
2. **어떻게 봤나** — 데이터·처리 규칙 요약(무엇을 지우고 무엇을 남겼는지, 그 이유).
3. **무엇을 발견했나** — 수치 + 그래프로 뒷받침되는 핵심 발견 2~4개(통계를 했다면 p·효과크기도).
4. **그래서 무엇을 하나** — 발견을 실행 가능한 제안으로(누구에게·무엇을·어떻게).
5. **한계** — 이 결론을 과신하면 안 되는 이유.

**숫자를 말로 옮기는 법** — ③ 발견을 쓸 때는 항상 이 순서를 따르세요.
1. **숫자 재진술** — 무엇을 비교했고 값이 얼마였는지 있는 그대로 적는다.
2. **유의성 + 효과크기** — p-value(우연인지)와 Cohen's d·η²·Cramér's V(크기)를 함께 적는다.
3. **방향** — 그래서 무엇이 더/덜 한지 한 문장으로 정리한다.

이때 두 가지를 절대 헷갈리지 마세요 — **유의 ≠ 실질**(p 가 작다고 차이가 큰 것은 아님, 특히 표본이 클 때)과 **상관 ≠ 인과**(같이 움직인다고 한쪽이 다른 쪽의 원인은 아님)입니다.

**좋은 예 / 나쁜 예**

| | 나쁜 예 | 좋은 예 |
| --- | --- | --- |
| 발견 | "온라인이 잘 팔린다" | "온라인 평균 단가가 오프라인보다 **약 1.3배**(0.0299 vs 0.0228) 높고, 이 차이는 통계적으로 유의하며 효과크기(Cohen's d=0.376)도 작은~중간 수준이다." |
| 제안 | "마케팅을 더 하자" | "단가가 높은 온라인 채널에 매출 상위 상품군(상의류) 노출을 강화하고, 가입 대기(PRE-CREATE) 고객 대상 온라인 온보딩을 설계한다." |
| 한계 | (언급 없음) | "이 데이터는 **Kaggle 이 공개한 표본**이라 전체 H&M 고객이 아니고, **관찰 자료라 인과가 아니며**, `price` 는 실제 화폐가 아닌 **정규화된 상대값**이다." |

**한계·해석 규율 — 어떤 결론을 쓰든 빠뜨리면 안 되는 여섯 가지**
- **매출의 정의(수량이 없다)**: 거래 표에 **수량(quantity) 컬럼이 없습니다.** 그래서 `price` 를 더한 값은 엄밀히 "매출액"이 아니라 **"구매 단가의 총합"** 입니다 — 리포트에 이 말을 그대로 적어 두세요. 게다가 이 값은 실제 화폐가 아니라 **0~1 사이로 정규화된 상대값**입니다(직접 `df['price'].min()`·`max()`·`mean()` 으로 확인해 보세요). "1,000원"처럼 절대 금액으로 읽거나 통화 단위를 붙여 쓰면 안 됩니다. 참고로 발제문에는 통화 단위를 언급하라고 적혀 있지만 이 데이터의 `price` 는 그 서술과 맞지 않습니다 — **문서와 데이터가 다를 때는 데이터를 믿고, 그 차이를 리포트에 적으세요.**
- **표본 편향**: 이 데이터는 Kaggle 이 공개한 **일부 표본**입니다 — 특히 **온라인(`sales_channel_id=2`) 기록이 상대적으로 많이 담겨** 있어 채널 비중이 실제보다 온라인 쪽으로 왜곡돼 있을 수 있습니다. 그래서 채널·집단을 비교할 때는 **건수 그대로가 아니라 비율(%)이나 고객당 평균 구매액(ARPU)으로 보정**해 비교하세요.
- **상관 ≠ 인과**: 연령대·채널·멤버십과 단가 사이의 관계는 원인이 아니라 **관찰된 결과**입니다. "20대 단가가 높다 → 20대를 집중 타깃으로" 같은 도약을 하지 마세요(무작위 배정 실험이 아니므로 "온라인이라서 비싸졌다"도 말할 수 없습니다). 제안은 "이 세그먼트에서 **검증해 볼 가치가 있다**" 수준으로 씁니다.
  · 참고로 발제문은 이 대목에서 **성별**을 예로 들며 "20대 여성" 을 언급하지만, 이 데이터의 고객 테이블에는 **성별 컬럼이 아예 없습니다**(`customer_id`·`FN`·`Active`·`club_member_status`·`fashion_news_frequency`·`age` 6개). 발제문에 적힌 변수가 실제로 있는지 **직접 확인하고** 없으면 "이 표본에는 성별 정보가 없어 분석하지 못했다"고 한계에 적으세요.
- **시즌성**: 특정 월의 급증을 **"성장"으로 단정하지 마세요.** 블랙프라이데이·연말연시 같은 이벤트가 반영돼 있을 수 있습니다. 추세를 말하려면 여러 기간을 비교하거나 롤링 평균처럼 노이즈를 줄인 뒤 말하세요.
- **인기 ≠ 트렌드 선도**: 상품군·색상의 매출 상위는 **공급량·재고 정책·프로모션**의 결과일 수 있습니다. "상의류가 1위"는 사실이지만 "상의류가 트렌드를 이끈다"는 이 데이터로 말할 수 없습니다.
- **인플레이션·환율 미반영**: 시점별 물가·환율 차이가 보정되지 않았으므로 **절대적 추세 해석은 제한적**입니다 — 같은 시점 안에서의 **상대 비교**에 초점을 두세요.

**리포트 템플릿** — 아래 빈칸을 채우세요.

In [ ]:
# 코드가 아니라 서술입니다 — 아래 마크다운 셀에 리포트를 작성하세요.

*(① 무엇을 물었나)*

*(② 어떻게 봤나 — 데이터·처리 규칙 요약)*

*(③ 무엇을 발견했나 — 수치+그래프 근거, 통계를 했다면 p·효과크기도)*

*(④ 그래서 무엇을 하나 — 실행 가능한 제안)*

*(⑤ 한계 — 위 여섯 가지(매출의 정의·표본 편향·상관≠인과·시즌성·인기≠트렌드·물가·환율)를 반드시 포함)*

## 5. AI 협업 검증
리포트 초안 작성에 AI(챗봇형 도구)를 쓰면 빠르지만, AI 는 **숫자·인과·연관 강도를 그럴듯하게 지어내거나 과장할 수 있습니다(환각)**. 여기서는 실제 API 호출 없이, **이미 오류가 심어진 AI 초안을 직접 재계산해 잡아내는 연습**을 합니다.

**① 좋은 프롬프트 4요소** — 맥락(무슨 데이터·무슨 분석인지) + 수치(내가 가진 정확한 값) + 요청(무엇을 써 달라는지) + 형식(불릿·길이 등), 그리고 "**준 수치 외의 값은 지어내지 말 것**"을 마지막에 덧붙이면 환각을 줄일 수 있습니다.

**② AI 초안 감사 — 아래 초안에는 오류 3개가 숨어 있습니다**

> H&M 거래 데이터를 분석한 결과, **온라인 채널의 매출 비중이 약 90%에 달합니다.** 이는 온라인이 이미 압도적인 주력 채널임을 보여줍니다. 그렇다면 **오프라인 고객을 온라인으로 전환시키면 매출이 곧바로 증가할 것입니다.** 한편 상품군별로 보면 상의류(Garment Upper body)가 매출 1위인데, 이는 **20대가 상의를 집중적으로 구매하기 때문**입니다.

**요구사항** — 아래 답안 셀에서 세 주장을 각각 **코드로 재계산**해 대조하고, 대조표를 완성한 뒤 초안을 고쳐 쓰세요. 자가채점이 값을 확인하므로 **변수 이름은 아래에 적힌 그대로** 쓰고, 세 값 모두 **0~1 사이의 비율**로 두세요(백분율로 바꾸지 마세요 — 출력할 때만 100을 곱하면 됩니다).

| # | 재계산할 것 | 변수명 | 쓸 컬럼·값 |
| --- | --- | --- | --- |
| 1 | 온라인 채널의 **실제 매출 비중**(거래 건수가 아니라 **수익**=`price` 합 기준) | `online_share` | `sales_channel_id` 별 `price` 합에서 채널 `2` 의 몫 |
| 2 | 채널별 **가입 대기(PRE-CREATE) 회원 비중** | `offline_precreate` · `online_precreate` | `sales_channel_id` × `club_member_status` 교차표를 **행 기준 비율**로(`pd.crosstab(..., normalize='index')`) 만든 뒤 `'PRE-CREATE'` 열에서 채널 `1`·`2` 값 |
| 3 | 상의 구매 중 20대 비중 vs 전체 거래 중 20대 비중 | `share_upper_20s` · `share_all_20s` | `product_group_name == 'Garment Upper body'` 로 걸러낸 뒤 `age_group == 20` 의 비율, 그리고 전체 `df` 에서 같은 비율 |

숫자를 구한 뒤 **각 주장을 판정**하세요 — ① 실제 매출 비중은 초안의 "약 90%"와 얼마나 다른가? ② 두 채널의 PRE-CREATE 비중이 서로 크게 다르다면 그것은 "채널이 무작위로 배정되지 않았다"는 신호인데, 그렇다면 "전환하면 매출이 는다"는 **인과**를 이 데이터로 말할 수 있는가? ③ 상의 구매 중 20대 비중은 전체보다 높은가, 낮은가? 그 결과가 "20대가 집중 구매해서"라는 설명을 뒷받침하는가?

> ⚠️ **여기만 예외적으로 자가채점이 있습니다** — 재계산 자체가 맞는지 확인하기 위해서입니다(분석·서술은 여전히 채점하지 않습니다). 허용오차: **비중은 ±0.5%p**, 나머지는 **부등호 비교**입니다.

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 재계산이 맞는지만 확인합니다(서술은 채점하지 않습니다).
assert abs(online_share * 100 - 74.7) < 0.5, '온라인 매출 비중 재계산을 다시 확인하세요'
assert offline_precreate < online_precreate, ('PRE-CREATE 는 오프라인보다 온라인에 훨씬 몰려 있어야 '
                                              '합니다(채널이 무작위 배정이 아니라는 근거)')
assert share_upper_20s < share_all_20s, ('상의 구매자 중 20대 비중이 전체보다 낮아야 합니다 '
                                         '("20대 집중구매" 주장이 근거 없음을 보여주는 지점)')
print('✅ 재계산 확인 완료')

**대조표**

| AI 초안 주장 | 재계산 값 | 판정 | 고친 문장 |
| --- | --- | --- | --- |
| "온라인 매출 비중이 약 90%" | *(위에서 구한 값)* | *(❌/✅)* | *(고친 문장)* |
| "전환시키면 매출이 곧바로 증가" | *(위에서 구한 값)* | *(❌/✅)* | *(고친 문장)* |
| "20대가 집중 구매해서 상의가 1위" | *(위에서 구한 값)* | *(❌/✅)* | *(고친 문장)* |

*(③ 반영 원칙을 한 문장으로 서술하세요)*

## 6. 제출 체크리스트
제출 전, 발제문의 **필수 목표**를 다 담았는지 스스로 확인하세요(체크만 하는 절이라 자가채점은 없습니다).

| # | 항목 | 어디서 |
| --- | --- | --- |
| 1 | 비즈니스 목표 세우기 | `01_데이터이해_전처리` 1장 |
| 2 | 데이터 소스 설명 | `01_데이터이해_전처리` 2장 |
| 3 | [EDA] 규모 파악 | `01_데이터이해_전처리` |
| 4 | [EDA] 타입과 기술통계 | `01_데이터이해_전처리` 2장 + `02_데이터_EDA` |
| 5 | [전처리] 테이블 결합 | `01_데이터이해_전처리` |
| 6 | [전처리] 규칙 수립·실행(이유 포함) | `01_데이터이해_전처리` |
| 7 | [분석·시각화] 기준 컬럼 비교 + 3개 이상 그림 | `02_데이터_EDA` |
| 8 | [인사이트] 최소 1개 이상, 수치·그래프·해석 | `02_데이터_EDA` + 이 노트북 4절 |

통계(2·3절)를 했다면 **덤**입니다 — 위 8개 항목에는 포함되지 않지만 리포트를 더 탄탄하게 만듭니다.